# NeuroContour

**Brain tumor segmentation with a U-Net, tuned in stages · Jacob Russ · MTH 4320 · Project 2**

## Introduction

**The question.** Can a U-Net accurately outline a whole glioma on a single 2D MRI slice — and can a staged search, each experiment chosen from the last one's results, beat one baseline configuration?

**The data.** The BraTS 2020 dataset, from the Brain Tumor Segmentation (BraTS) challenge, accessed through Kaggle: pre-operative glioma MRI gathered from 19 institutions. Each labeled case is one patient with four co-registered MRI volumes (T1, T1ce, T2, FLAIR) at 240×240×155, plus a manual tumor outline approved by neuro-radiologists. The outline marks three sub-regions (labels 1, 2, and 4); this project collapses them into one binary whole-tumor mask. Of the 369 labeled cases, this project uses the first 75, and keeps only the slices that carry at least 50 tumor pixels — 4,724 slices in total.

**The plan.** Part 1 prepares the data and splits it by patient (52 train / 11 validation / 12 test), so no brain ever appears in two splits. Part 2 builds the U-Net and its training code. Part 3 runs the experiments in stages — learning rate first, then a baseline, then one change at a time (loss, augmentation), then an ablation that removes the skip connections. Part 4 touches the test set exactly once, with the winning configuration. The score throughout is the Dice coefficient, the overlap between the predicted and expert outlines, averaged per patient.

## Part 1 — Data preparation

Everything in this part exists to make the later results believable: the right 75 cases, a mask definition that matches the question, slices worth training on, a split that cannot leak, and one dishonest baseline exposed before any model is trained.

**Code cell 1 — Imports and fixed settings**

Bring in the libraries the whole notebook uses (pathlib, random, numpy, nibabel, torch, matplotlib) and set the fixed settings every later cell shares: where the data lives, the 75 case numbers, the slice-keeping rule, image size, batch size, seed, and epoch limits. Settings are set once, here, so every run below is reproducible.

**Code cell 2 — Seed and device**

Fix the random seeds (Python, numpy, torch) and pick the compute device — MPS on an Apple-silicon Mac, CUDA if present, otherwise CPU — then print both so the choice is visible in the output.

**Code cell 3 — Find the 75 cases**

Starting from the data root, build the sorted list of case folders for BraTS20_Training_001 through BraTS20_Training_075 and print how many were found, plus the first and last names.

**Code cell 4 — Inspect one case folder**

List the files inside the first case folder and print them. A complete case has exactly five files: the four modality scans and the one segmentation file. Note the exact file suffix your copy uses, because the loading code has to match it.

**Code cell 5 — Load one case**

Load all five volumes of the first case with nibabel and print each array's shape. Every volume should read (240, 240, 155).

**Code cell 6 — Look at the labels**

Print the distinct label values present in the first case's segmentation, and the number of tumor pixels (any label above zero) on a few example slices. The values should come only from 0, 1, 2, and 4.

**Code cell 7 — Build the slice index**

Loop over all 75 cases. For each one, load only the segmentation volume, count tumor pixels on every slice, and record (case, slice number) for every slice with at least 50 tumor pixels. This list is the dataset every later cell draws from.

**Code cell 8 — Check the slice index**

Print the total number of kept slices and the smallest, largest, and average kept-slice counts per case. The total should be 4,724.

**Code cell 9 — Split the patients**

Shuffle the 75 case names with the project seed and divide them into 52 training, 11 validation, and 12 test cases. Store the three lists. The split is by patient, never by slice.

**Code cell 10 — Check the split**

Prove the split is clean: print the three patient counts, confirm no case appears in more than one list, and count how many kept slices fall in each split. The three slice counts must add back up to the index total.

**Code cell 11 — The honest baseline**

Score a predictor that outputs 'no tumor anywhere' against the real validation masks: compute its Dice and its pixel accuracy. Dice should be exactly 0 while accuracy sits near 98% — the reason this project never headlines accuracy.

## Part 2 — The model and how it trains

The model is a U-Net: an encoder that compresses the slice through widths 32–256 into a 512-wide bottleneck, and a mirrored decoder that expands it back to full size, with skip connections carrying fine boundary detail across. It is built with a switch that removes those skip connections, so Part 3 can measure what they are worth. Training uses the Adam optimizer; two losses are on trial — binary cross-entropy with logits, and Dice loss.

**Code cell 12 — Loading helpers**

Write the two small helpers every example passes through: one resizes a scan slice to 128×128 smoothly and normalizes it using that slice's own mean and standard deviation; the other resizes a mask slice with nearest-neighbor so it stays strictly 0/1.

**Code cell 13 — The slice dataset**

Write the PyTorch Dataset that serves one example: given a (case, slice) pair from the index, it loads the four modality slices and the mask slice, runs them through the helpers, stacks the scans into a (4, 128, 128) tensor, and — only when flips are switched on — flips scan and mask together left-right. It returns the scan tensor and a (1, 128, 128) mask tensor.

**Code cell 14 — Check one example**

Pull one training example from the dataset and one batch from a DataLoader, and print their shapes. One example is (4, 128, 128) with mask (1, 128, 128); a batch of 8 is (8, 4, 128, 128) with mask (8, 1, 128, 128).

**Code cell 15 — The building block**

Write the repeated block of the network — two 3×3 convolutions, each followed by batch normalization and a ReLU — as one small module that every stage of the U-Net reuses.

**Code cell 16 — The U-Net**

Assemble the full model from the block: four encoder stages with max-pooling between them, the bottleneck, four decoder stages with transposed convolutions, and a final 1×1 convolution to one output channel. When the skip switch is on, each decoder stage receives the matching encoder output joined along the channel axis; when it is off, it does not.

**Code cell 17 — Check the model**

Send a dummy batch of shape (1, 4, 128, 128) through the model twice — skips on and skips off — and print the output shape both times, (1, 1, 128, 128), plus the parameter count for each version. The no-skip version should have fewer parameters.

**Code cell 18 — Dice score and Dice loss**

Write the two functions the project is judged by: a Dice score between a thresholded prediction and a mask, and a soft Dice loss on probabilities for training. Sanity-check both on tiny arrays where the right answers are known before trusting them on the model.

**Code cell 19 — One epoch, and one validation pass**

Write the two halves of training: a function that trains the model for one pass over a loader (forward, loss, backward, step) and returns the average loss, and a function that runs the model over the validation loader with gradients off and returns the mean per-slice Dice.

**Code cell 20 — The run function**

Write the function every experiment below calls. Given a configuration — learning rate, loss choice, flips on or off, skips on or off, epoch cap — it builds the model, datasets, loaders, and optimizer, trains with early stopping on validation Dice, keeps the best-scoring weights, records the run in the shared results list, and returns the best validation Dice and the saved weights' location.

## Part 3 — Experiments: one change at a time

The experiments run in stages, and each stage's settings are chosen from the stage before it — nothing is tuned by brute force. Stage 1 picks the learning rate with three short runs; the winner is stored as BEST_LR and used by every run after it. Stage 2 trains the full baseline. Stages 3 and 4 each change exactly one thing against the current winner. Stage 5 removes the skip connections to test the architecture itself. Every run, good or bad, stays in the results, and the stage-by-stage table at the end of this part is the record the test run is chosen from.

**Code cell 21 — Stage 1 — three short learning-rate runs**

Run the run function three times at learning rates 3×10⁻⁴, 1×10⁻³, and 3×10⁻³, each for a few epochs on the first 600 training slices only. Everything else stays at the baseline settings.

**Code cell 22 — Read the learning-rate runs**

Print the three short runs side by side — name, learning rate, best validation Dice — and record which learning rate won. That rate is used by every full run after this cell.

**Code cell 23 — Stage 2 — the baseline run**

Train the full model at the winning learning rate with BCE loss, flips off, skips on, and early stopping. This run is the reference every later comparison is measured against.

**Code cell 24 — Stage 3 — change the loss**

Repeat the baseline run with exactly one change: Dice loss in place of BCE. Same learning rate, same data, same everything else.

**Code cell 25 — Stage 4 — change the augmentation**

Repeat the current winning configuration with exactly one change: left-right flip augmentation switched on.

**Code cell 26 — Stage 5 — the ablation**

Repeat the current winning configuration with exactly one change: the skip connections switched off. This run prices the U-Net's defining feature.

**Code cell 27 — The results table**

Build one table of every full run — stages 2 through 5 — with its configuration and best validation Dice, sorted best first, and print it. Nothing is deleted: failed and losing runs stay visible.

## Part 4 — Final test results

The test patients have not been touched by any decision so far. This part loads the winning configuration's saved weights once and scores them on the 12 test patients — per patient first, then averaged, the way the project agreed to report it.

**Code cell 28 — Score the test patients**

Rebuild the winning configuration, load its saved best weights, and compute the Dice for every kept test slice, grouped by patient. Store each patient's mean Dice in a dictionary.

**Code cell 29 — Summarize the test results**

Print the final numbers: each test patient's mean Dice, and the overall mean, median, lowest, and highest across the 12 patients. Beside them, print the honest baseline's test Dice — 0 — for scale.

**Code cell 30 — Overlay figures**

For three test slices — one of the best-scoring patients, one near the middle, one of the worst — plot the FLAIR slice, the expert outline, the model's outline, and the two outlines overlaid. Save the figure.

**Code cell 31 — Wrong-pixel figures**

For the worst test slice, build the wrong-pixel map: pixels the model missed and pixels it added, shown in two colors over the scan. Save the figure and print the missed and added counts.

## Data credit

BraTS 2020 data, from the Brain Tumor Segmentation (BraTS) challenge (organized by the Center for Biomedical Image Computing and Analytics, University of Pennsylvania, with the MICCAI society), accessed through Kaggle (mirror by Awsaf, CC0). Dataset citations: Menze et al., "The Multimodal Brain Tumor Image Segmentation Benchmark (BRATS)", IEEE Transactions on Medical Imaging 34(10), 2015 · Bakas et al., "Advancing The Cancer Genome Atlas glioma MRI collections with expert segmentation labels and radiomic features", Scientific Data 4:170117, 2017 · Bakas et al., "Identifying the Best Machine Learning Algorithms for Brain Tumor Segmentation, Progression Assessment, and Overall Survival Prediction in the BRATS Challenge", arXiv:1811.02629, 2018.